## 多头注意力

多头注意力需要 Wq、Wk、Wv、Wo 这 4 类训练参数。多头注意力不改变缩放点积注意力或加性注意力本身，而是在使用 q、k、v 之前，先用训练参数将其投影到其他维度，从而能捕捉超越原本 q、k、v 的特征。底层仍然使用这些注意力方法。

**改变输入参数**：多头注意力是在使用 q、k、v 之前，先使用训练参数将其投影到其他维度，从而能捕捉超越原本 q、k、v 的特征。

不同的注意力头通过各自可训练的 Wq、Wk、Wv 来保证自己能注意到不同内容和结构，得出各自的 value 聚合结果；Wo 则决定输出时如何组织各注意力计算出的 value。

将原始 query、key、value 投影到不同的表示子空间。输入维度可以各不相同；但投影后 Q_i 和 K_i 的最后一维必须相同（d_k），V_i 的最后一维（d_v）可以不同。

对于第 i 个头：

Q_i = Q_input · Wq_i
K_i = K_input · Wk_i
V_i = V_input · Wv_i

其中 Q_i 和 K_i 的最后一维必须相同，记为 d_k。

f 可以是加性注意力或者点积注意力，输入的参数是投影后的 Q_i、K_i、V_i，而不是原始的 query、key、value。

Head_i = f(Q_i, K_i, V_i)

最终：

MultiHead = Concat(Head_1, ..., Head_N) · Wo

### 注意

**单头注意力**：
使用 512 维特征计算一张 [L_q, L_k] 分数矩阵，
然后对每个 query 对应的 L_k 个分数做一次 softmax。

**8 头注意力**：
每个头使用自己的 64 维投影计算一张 [L_q, L_k] 分数矩阵，
共得到 8 张分数矩阵，每张矩阵独立做 softmax。
所以即使单头和多头的特征维度相同（512 = 8 × 64），它们的结果也不一样。

### batch 化

queries: [batch_size, seq_len_q, query_size]  # 逐词解码时 seq_len_q = 1
keys:    [batch_size, seq_len, key_size]
values:  [batch_size, seq_len, value_size]

改造成 batch 后，q、k、v 的 batch 维度值应该相等，k 和 v 的 seq_len 应该相等，k 和 v 应该以键值对的形式出现。
比如 q 有 2 个 batch，k 和 v 也要有 2 个 batch，每个 batch k 中有 3 个 key，则 v 中也要有 3 个 value。

In [2]:
import torch
from torch import nn
from attention_layers import DotProductAttention


class MultiHeadAttention(nn.Module):
    def __init__(self, query_size, key_size, value_size, hidden_size, head_count) -> None:
        super().__init__()
        self.wq = nn.Linear(query_size, hidden_size * head_count)
        self.wk = nn.Linear(key_size, hidden_size * head_count)
        self.wv = nn.Linear(value_size, hidden_size * head_count)
        self.wo = nn.Linear(hidden_size * head_count, hidden_size)
        self.head_count = head_count
        self.hidden_size = hidden_size
        self.attention = DotProductAttention()

        # queries: [batch_size, seq_len_q, query_size]
        # keys:    [batch_size, seq_len_k, key_size]
        # values:  [batch_size, seq_len_k, value_size]
    def forward(self, queries, keys, values):
        prj_q = self.wq(queries)
        prj_k = self.wk(keys)
        prj_v = self.wv(values)
        print("prj_q.shape", prj_q.shape)

        # 将
        prj_q = self.split_heads_for_attention(prj_q)
        print("prj_q.shape", prj_q.shape)
        prj_k = self.split_heads_for_attention(prj_k)
        prj_v = self.split_heads_for_attention(prj_v)


        preds = self.attention(prj_q, prj_k, prj_v)
        print("preds.shape", preds.shape)

        preds = self.combine_attention(preds)
        print("preds.shape", preds.shape)

        preds =self.wo(preds)
        print("preds.shape", preds.shape)

        return preds

    # 注意力分数的本质是标量，各种注意力分数公式输出的都是标量。
    # prj_q，prj_k原本的形状是 (batch, seq, hidden_size * head_count)
    # 计算注意力时要转换成 (batch * head_count, seq, hidden_size)
    # 因为每对（q，k）注意力分数实际上就是最后一维计算出的分数
    # 如果不拆分，那么分数算出来的结果就是把多头的hidden一起打来一个分
    # 而不是为每对（q，k）在每个head上都打分
    def split_heads_for_attention(self, tensor):
        tensor = tensor.reshape(tensor.shape[0], tensor.shape[1], self.head_count, -1)
        tensor = tensor.permute(0, 2, 1, 3)
        tensor = tensor.reshape(-1, tensor.shape[2], tensor.shape[3])
        return tensor


    def combine_attention(self, tensor):
        # tensor 的形状是(batch * head_count, seq, hidden_size)
        # 我们需要再将其转换成(batch, seq, hidden_size * head_count)
        # 这样wo矩阵对每个 query 位置，将所有 head 的输出拼接起来，再对这些 head 提取出的信息进行学习和线性混合。
        # 变成 (seq, hidden_size * head_count) 更符合一个query对应多个head的不同value了
        tensor = tensor.reshape(tensor.shape[0]//self.head_count, self.head_count, tensor.shape[1], tensor.shape[2])
        tensor = tensor.permute(0, 2, 1, 3)
        tensor = tensor.reshape(tensor.shape[0], tensor.shape[1], -1)
        return tensor





    

In [4]:
query_size = 4
key_size = 4
value_size = 3
batch_size =3 
seq_query = 2
seq_key =4
hidden_size = 5
head_count = 3
net = MultiHeadAttention(query_size, key_size, value_size, hidden_size, head_count)



n_train = 50
queries = torch.randn(batch_size, seq_query, query_size)
keys = torch.randn(batch_size, seq_key, key_size)


values = torch.randn(batch_size, seq_key, value_size)

net(queries, keys, values)



prj_q.shape torch.Size([3, 2, 15])
prj_q.shape torch.Size([9, 2, 5])
preds.shape torch.Size([9, 2, 5])
preds.shape torch.Size([3, 2, 15])
preds.shape torch.Size([3, 2, 5])


tensor([[[ 0.1390, -0.1993,  0.4024, -0.0639,  0.0693],
         [ 0.2119, -0.1549,  0.3309, -0.0914,  0.0437]],

        [[ 0.2231, -0.2632,  0.1670, -0.1222, -0.0873],
         [ 0.2266, -0.3005,  0.1603, -0.0826, -0.1361]],

        [[ 0.6183,  0.0700, -0.0050, -0.1768, -0.1703],
         [ 0.6116,  0.0371, -0.0230, -0.1241, -0.2048]]],
       grad_fn=<ViewBackward0>)